### Hướng dẫn chạy:
- Cell đầu tiên có **4 biến** được phép chỉnh sửa là "MODEL_NAME", "DATA_NAME", "PRED_LEN" và "DATASET_DIR". Các giá trị có thể nhận của các biến "MODEL_NAME", "DATA_NAME", "PRED_LEN" để ở phần comment code. "DATASER_DIR" đổi thành đường dẫn đến Kaggle Dataset

- Mỗi người để "MODEL_NAME" thành model được chia từ trước, và chạy tổng là 5 x 3 = 15 lần (5 giá trị "DATA_NAME" với 3 giá trị "PRED_LEN")

- Mỗi lần chỉ cần đổi biến rồi bấm "Save and Run all" là được, sau khi chạy xong sẽ có tổng 4 file có đuôi csv (lưu loss train val), npz (lưu dự đoán tập test) và pt/pkl (lưu best state model). Các file có quy ước đặt tên dựa theo "MODEL_NAME", "DATA_NAME", "PRED_LEN" nên chỉ cần tải về thôi, không cần làm gì thêm. Chạy đủ 15 lần là được 45 file, toàn bộ phải up thẳng hết lên thư mục checkpoints trên github (không cần tạo thêm thư mục con hay gì cả)

- Ngoài ra chạy xong thì cell cuối có print metric đánh giá, cop rồi dán vào sheet thống kê kết quả là được

In [ ]:
GITHUB_REPO = "https://github.com/minhE10/time-series-prediction.git"

CLONE_DIR = "/kaggle/working/ts-project"                       

# arima | itransformer | timemixer | tsmamba | xgboost
MODEL_NAME = "timemixer"

# sunspots | appliances_energy | beijing_air_quality | hanoi_air_quality | bitcoin
DATA_NAME = "bitcoin"
SEQ_LEN = 48           # lookback window (fixed)
PRED_LEN = 12          # forecast horizon: 12 | 24 | 48
NUM_EPOCHS = 200         
SCALER = "standard"

# change to the right path
DATASET_DIR = "/kaggle/input/"
DATASET_PATHS = {
    "sunspots": f"{DATASET_DIR}/sunspots_dataset.csv",
    "appliances_energy": f"{DATASET_DIR}/appliances_energy_dataset.csv",
    "beijing_air_quality": f"{DATASET_DIR}/beijing_air_quality_dataset.csv",
    "hanoi_air_quality": f"{DATASET_DIR}/hanoi_air_quality_dataset.csv",
    "bitcoin": f"{DATASET_DIR}/bitcoin_dataset.csv"
}

# output directory for all saved artifacts
OUT_DIR = "/kaggle/working/outputs"

In [ ]:
import subprocess, sys
subprocess.run(["git", "clone", "--depth", "1", GITHUB_REPO, CLONE_DIR], check=True)
sys.path.insert(0, CLONE_DIR)

from setup import set_seed, clear_memory
SEED_WORKER, G, DEVICE = set_seed(42)

In [ ]:
from data_loader import TimeSeriesDataset

dataset = TimeSeriesDataset(
    name=DATA_NAME, path=DATASET_PATHS[DATA_NAME],
    seq_len=SEQ_LEN, pred_len=PRED_LEN,
    batch_size=32, worker_init_fn=SEED_WORKER, generator=G,
)
dataset.apply_scaling(SCALER)
train_loader, val_loader, test_loader = dataset.get_loaders()

print(f"Dataset : {DATA_NAME}  |  Features: {dataset.n_features}  Targets: {dataset.n_targets}")
print(f"Windows : train={len(dataset.train_dataset)}  val={len(dataset.val_dataset)}  test={len(dataset.test_dataset)}")

In [ ]:
from src.factory import build_model, build_trainer

model = build_model(MODEL_NAME, dataset, SEQ_LEN, PRED_LEN)
trainer = build_trainer(MODEL_NAME, model, dataset, train_loader, val_loader, test_loader, DEVICE)

if model is not None and hasattr(model, 'parameters'):
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Model: {MODEL_NAME}  |  Parameters: {n_params:,}")

In [ ]:
history = trainer.fit(NUM_EPOCHS)

In [ ]:
from plot import plot_loss, plot_predictions

trainer.evaluate(scaled=False, title=f"{MODEL_NAME.upper()} — {DATA_NAME}")

plot_loss(history, title=f"{MODEL_NAME.upper()} on {DATA_NAME}")

y_true, y_pred = trainer.predict()
plot_predictions(y_true, y_pred, dataset.target_cols,
                 title=f"{MODEL_NAME.upper()} on {DATA_NAME} (Test)")

In [ ]:
from evaluate import save_history, save_checkpoint, save_predictions

y_true, y_pred = trainer.predict()

save_history(history, MODEL_NAME, PRED_LEN, DATA_NAME, OUT_DIR)
save_checkpoint(trainer, MODEL_NAME, PRED_LEN, DATA_NAME, OUT_DIR)
save_predictions(y_true, y_pred, dataset.target_cols, MODEL_NAME, PRED_LEN, DATA_NAME, OUT_DIR)

print(f"\nAll artifacts saved to: {OUT_DIR}/")
print(f"  Prefix: {MODEL_NAME}_{PRED_LEN}_{DATA_NAME}")

In [ ]:
from evaluate import load_history, load_predictions
from plot import plot_loss, plot_predictions

history_loaded = load_history(MODEL_NAME, PRED_LEN, DATA_NAME, OUT_DIR)
y_true_loaded, y_pred_loaded, target_cols_loaded = load_predictions(MODEL_NAME, PRED_LEN, DATA_NAME, OUT_DIR)

plot_loss(history_loaded, title=f"{MODEL_NAME.upper()} on {DATA_NAME}")
plot_predictions(y_true_loaded, y_pred_loaded, target_cols_loaded,
                 title=f"{MODEL_NAME.upper()} on {DATA_NAME} (Test)")